In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_controles AS
SELECT 'bronze_con_datos' AS control,
       (SELECT least(
          (SELECT count(*) FROM workspace.tp_futbol.bronze_partidos_europa),
          (SELECT count(*) FROM workspace.tp_futbol.bronze_partidos_arg),
          (SELECT count(*) FROM workspace.tp_futbol.bronze_estadios_wikidata),
          (SELECT count(*) FROM workspace.tp_futbol.bronze_estadios_manual),
          (SELECT count(*) FROM workspace.tp_futbol.bronze_equipos_equivalencia),
          (SELECT count(*) FROM workspace.tp_futbol.bronze_restricciones_publico))) AS valor,
       '> 0' AS esperado
UNION ALL
SELECT 'partidos_conservados_bronze_a_silver',
       (SELECT count(*) FROM workspace.tp_futbol.bronze_partidos_europa)
     + (SELECT count(*) FROM workspace.tp_futbol.bronze_partidos_arg)
     - (SELECT count(*) FROM workspace.tp_futbol.silver_partidos)
     - (SELECT count(*) FROM workspace.tp_futbol.silver_partidos_cuarentena), '= 0'
UNION ALL
SELECT 'partido_id_duplicados',
       (SELECT count(*) - count(DISTINCT partido_id) FROM workspace.tp_futbol.silver_partidos), '= 0'
UNION ALL
SELECT 'pct_cuarentena_partidos_x1000',
       (SELECT round(1000 * count(*) / (SELECT count(*) FROM workspace.tp_futbol.silver_partidos))
        FROM workspace.tp_futbol.silver_partidos_cuarentena), '< 10'
UNION ALL
SELECT 'equipos_sin_estadio',
       (SELECT count(*) FROM (
          SELECT liga, local AS equipo FROM workspace.tp_futbol.silver_partidos
          UNION SELECT liga, visitante FROM workspace.tp_futbol.silver_partidos) p
        LEFT ANTI JOIN workspace.tp_futbol.silver_equipos e USING (liga, equipo)), '= 0'
UNION ALL
SELECT 'restricciones_en_cuarentena',
       (SELECT count(*) FROM workspace.tp_futbol.silver_restricciones_cuarentena), '= 0'
UNION ALL
SELECT 'gold_igual_a_silver',
       (SELECT count(*) FROM workspace.tp_futbol.gold_partidos)
     - (SELECT count(*) FROM workspace.tp_futbol.silver_partidos), '= 0'
UNION ALL
SELECT 'gold_sin_regimen_o_distancia',
       (SELECT count_if(regimen_publico IS NULL OR distancia_km IS NULL) FROM workspace.tp_futbol.gold_partidos), '= 0'
UNION ALL
SELECT 'probabilidades_no_suman_1',
       (SELECT count_if(abs(prob_local + prob_empate + prob_visitante - 1) > 1e-9)
        FROM workspace.tp_futbol.gold_partidos WHERE prob_local IS NOT NULL), '= 0';

SELECT *,
       CASE esperado WHEN '> 0'  THEN valor > 0
                     WHEN '= 0'  THEN valor = 0
                     WHEN '< 10' THEN valor < 10 END AS ok
FROM v_controles;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.tp_futbol.validacion_auditoria (
  ejecutado_en TIMESTAMP, bronze_partidos BIGINT, silver_partidos BIGINT,
  cuarentena_partidos BIGINT, gold_partidos BIGINT, controles_fallidos BIGINT
);

INSERT INTO workspace.tp_futbol.validacion_auditoria
SELECT current_timestamp(),
       (SELECT count(*) FROM workspace.tp_futbol.bronze_partidos_europa) + (SELECT count(*) FROM workspace.tp_futbol.bronze_partidos_arg),
       (SELECT count(*) FROM workspace.tp_futbol.silver_partidos),
       (SELECT count(*) FROM workspace.tp_futbol.silver_partidos_cuarentena),
       (SELECT count(*) FROM workspace.tp_futbol.gold_partidos),
       (SELECT count_if(NOT CASE esperado WHEN '> 0' THEN valor > 0 WHEN '= 0' THEN valor = 0 WHEN '< 10' THEN valor < 10 END)
        FROM v_controles);

SELECT assert_true(
  (SELECT controles_fallidos FROM workspace.tp_futbol.validacion_auditoria ORDER BY ejecutado_en DESC LIMIT 1) = 0,
  'Fallaron controles de calidad: revisar v_controles'
) AS validacion;